# 02 — Latency Analysis

This notebook answers the primary research question:
**Does the MediatR dispatch layer in .NET add measurable latency compared to the
direct goroutine-channel model in Go?**

We run Mann-Whitney U tests, compute Cliff's delta effect sizes, derive 95%
bootstrap confidence intervals on the mean difference, and visualise the
full latency distribution as an empirical CDF.

## Imports and setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import (
    SERVICES, SERVICE_LABELS, SERVICE_COLORS,
    load_csv, describe_series, set_plot_style,
)
from statistical_tests import (
    _compute_test_result, run_latency_tests, format_results_table,
)
from generate_charts import fig01_latency_percentiles, fig02_latency_timeseries, fig06_latency_cdf

set_plot_style()
%matplotlib inline

## Load latency data

We load the P95 and P99 series (one row per Prometheus scrape interval).
Aligning both services on the shared time index gives us paired observations.

In [ ]:
p95_dn = load_csv('latency_p95_dotnet.csv')['value_ms']
p95_go = load_csv('latency_p95_go.csv')['value_ms']
p99_dn = load_csv('latency_p99_dotnet.csv')['value_ms']
p99_go = load_csv('latency_p99_go.csv')['value_ms']

print('P95 .NET — samples:', len(p95_dn), '| P95 Go — samples:', len(p95_go))
print('P99 .NET — samples:', len(p99_dn), '| P99 Go — samples:', len(p99_go))

## Descriptive statistics table

Before hypothesis testing we inspect the distribution shape.
Skewed P99 vs P95 ratios indicate tail events worthy of further investigation.

In [ ]:
rows = []
for label, series in [
    ('.NET P95', p95_dn), ('Go P95', p95_go),
    ('.NET P99', p99_dn), ('Go P99', p99_go),
]:
    d = describe_series(series)
    rows.append([label, f"{d['mean']:.2f}", f"{d['std']:.2f}",
                 f"{d['p50']:.2f}", f"{d['p95']:.2f}", f"{d['p99']:.2f}",
                 f"{d['max']:.2f}"])

headers = ['Series', 'Mean', 'Std', 'P50', 'P95', 'P99', 'Max']
pd.DataFrame(rows, columns=headers)

## Mann-Whitney U — P95 latency

We use the non-parametric Mann-Whitney U test because latency distributions are
typically right-skewed and do not meet the normality assumption required by
a Student's t-test.

In [ ]:
r95 = _compute_test_result(
    p95_dn.dropna().to_numpy(), p95_go.dropna().to_numpy(),
    'P95 Latency', 'ms', n_bootstrap=10_000,
)
print(f'U statistic : {r95.u_stat:.0f}')
print(f'p-value     : {r95.p_value:.6f}  ({"significant" if r95.significant else "not significant"} at α=0.05)')
print(f'.NET mean   : {r95.dotnet_mean:.2f} ms')
print(f'Go mean     : {r95.go_mean:.2f} ms')
print(f'Difference  : {r95.mean_diff:+.2f} ms  (.NET − Go)')
print(f'Winner      : {r95.winner.upper()}')

## Mann-Whitney U — P99 latency

In [ ]:
r99 = _compute_test_result(
    p99_dn.dropna().to_numpy(), p99_go.dropna().to_numpy(),
    'P99 Latency', 'ms', n_bootstrap=10_000,
)
print(f'U statistic : {r99.u_stat:.0f}')
print(f'p-value     : {r99.p_value:.6f}  ({"significant" if r99.significant else "not significant"} at α=0.05)')
print(f'.NET mean   : {r99.dotnet_mean:.2f} ms')
print(f'Go mean     : {r99.go_mean:.2f} ms')
print(f'Difference  : {r99.mean_diff:+.2f} ms  (.NET − Go)')
print(f'Winner      : {r99.winner.upper()}')

## Bootstrap confidence interval on mean difference

10 000 bootstrap resamples give us a 95% CI on the mean latency difference
(dotnet − go).  If the CI excludes 0 we have strong evidence of a consistent
directional difference beyond statistical noise.

In [ ]:
print(f'P95 95% CI on mean diff: [{r95.ci_low:+.2f}, {r95.ci_high:+.2f}] ms')
print(f'P99 95% CI on mean diff: [{r99.ci_low:+.2f}, {r99.ci_high:+.2f}] ms')

rng = np.random.default_rng(42)
dn_arr = p95_dn.dropna().to_numpy()
go_arr = p95_go.dropna().to_numpy()
bs_diffs = np.array([
    rng.choice(dn_arr, size=len(dn_arr), replace=True).mean()
    - rng.choice(go_arr, size=len(go_arr), replace=True).mean()
    for _ in range(5_000)
])

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(bs_diffs, bins=60, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline(r95.ci_low,  color='crimson', linestyle='--', label='2.5%')
ax.axvline(r95.ci_high, color='crimson', linestyle='--', label='97.5%')
ax.axvline(0, color='black', linewidth=1, label='No difference')
ax.set_xlabel('Bootstrap mean difference (ms)')
ax.set_ylabel('Count')
ax.set_title('Bootstrap distribution of P95 mean diff (.NET − Go)')
ax.legend()
plt.tight_layout()
plt.show()

## Effect size — Cliff's delta

Cliff's delta measures the probability that a random .NET observation exceeds
a random Go observation.  0.5 = no difference; thresholds: <0.147 negligible,
<0.33 small, <0.474 medium, ≥0.474 large.

In [ ]:
print(f'P95 Cliff\'s delta: {r95.effect_size:.4f}  → {r95.effect_label} effect')
print(f'P99 Cliff\'s delta: {r99.effect_size:.4f}  → {r99.effect_label} effect')

## Figure 1: latency percentile bar chart

In [ ]:
fig01_latency_percentiles()

## Figure 2: latency time series

In [ ]:
fig02_latency_timeseries()

## Figure 6: empirical CDF

In [ ]:
fig06_latency_cdf()

## Interpretation

**Fill in after running with real data.**

Template:

> The Go service achieved **X% lower P95 latency** (Mann-Whitney U, p=XXXX,
> Cliff's delta=X.XX, **Y effect**).  The 95% bootstrap CI on the mean difference
> was [+A ms, +B ms], which excludes zero — confirming a consistent directional
> advantage for Go under steady-state load.
>
> At P99 the gap widens to Z ms, suggesting that the MediatR dispatch pipeline
> introduces occasional tail latency likely attributable to .NET thread-pool
> scheduling under contention.
>
> **Dissertation answer (RQ1):** The MediatR layer adds approximately X ms to
> mean P95 latency (Y% overhead) — statistically significant but practically
> [negligible / notable] for workloads with an SLA of Z ms.